In [ ]:
import os
import random
from pathlib import Path
from io import BytesIO
import zipfile

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy import ndimage

In [ ]:
# ============================================================
# SLART V10: SAFE UPGRADE #2 (THRESHOLD 0.5 → 0.45)
# Expected improvement: +0.03 to +0.08 score
# ============================================================


# ------------------------------------------------------------
#  Config - SAME AS BEFORE except inference threshold change
# ------------------------------------------------------------
ROOT = Path("/kaggle/input/vesuvius-challenge-surface-detection")
TRAIN_IMG_DIR = ROOT / "train_images"
TRAIN_LABEL_DIR = ROOT / "train_labels"
TEST_IMG_DIR = ROOT / "test_images"

TRAIN_CSV = ROOT / "train.csv"
TEST_CSV = ROOT / "test.csv"

OUT_ZIP = Path("/kaggle/working/submission.zip")

MAX_TRAIN_VOLUMES = 15
PATCH_SIZE = 128
TRAIN_SAMPLES = 10000
VAL_SAMPLES = 1200
BATCH_SIZE = 4
EPOCHS = 6
LR = 1e-3

MIN_COMPONENT_VOXELS = 600
CLOSING_ITERS = 1

# SAFE UPGRADE #2 ----
THRESH = 0.45
# ---------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
#  Utilities: load/save volumes
# ============================================================

def load_stack(path: Path) -> np.ndarray:
    with Image.open(path) as tif:
        frames = [np.array(frame) for frame in ImageSequence.Iterator(tif)]
    return np.stack(frames).astype(np.float32)

def normalize_volume(vol: np.ndarray) -> np.ndarray:
    v = vol.astype(np.float32)
    return (v - v.mean()) / (v.std() + 1e-6)

def write_stack_to_zip(array3d: np.ndarray, zip_handle, name: str):
    pages = [Image.fromarray(s.astype(np.uint8)) for s in array3d]
    buffer = BytesIO()
    pages[0].save(buffer, format="TIFF", save_all=True, append_images=pages[1:])
    zip_handle.writestr(name, buffer.getvalue())

# ============================================================
#  Postprocessing
# ============================================================

def clean_mask(mask: np.ndarray) -> np.ndarray:
    cc_structure = ndimage.generate_binary_structure(3, 1)
    labeled, num = ndimage.label(mask, structure=cc_structure)

    if num == 0:
        return mask.astype(np.uint8)

    component_sizes = ndimage.sum(mask, labeled, index=np.arange(1, num + 1))
    keep_labels = np.where(component_sizes >= MIN_COMPONENT_VOXELS)[0] + 1
    cleaned = np.isin(labeled, keep_labels).astype(np.uint8)

    if CLOSING_ITERS > 0:
        closing_structure = np.zeros((3, 3, 3), dtype=np.uint8)
        closing_structure[1, :, :] = 1
        cleaned = ndimage.binary_closing(
            cleaned, structure=closing_structure, iterations=CLOSING_ITERS
        ).astype(np.uint8)

    return cleaned

# ============================================================
#  UNet (same as before)
# ============================================================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base_ch=32):
        super().__init__()

        self.down1 = DoubleConv(in_ch, base_ch)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(base_ch, base_ch * 2)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(base_ch * 2, base_ch * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.bottom = DoubleConv(base_ch * 4, base_ch * 8)

        self.up3 = nn.ConvTranspose2d(base_ch * 8, base_ch * 4, 2, 2)
        self.dec3 = DoubleConv(base_ch * 8, base_ch * 4)

        self.up2 = nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 2, 2)
        self.dec2 = DoubleConv(base_ch * 4, base_ch * 2)

        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, 2, 2)
        self.dec1 = DoubleConv(base_ch * 2, base_ch)

        self.out_conv = nn.Conv2d(base_ch, out_ch, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x4 = self.bottom(self.pool3(x3))

        x = self.up3(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.dec3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        return self.out_conv(x)

# ============================================================
# Dataset (same)
# ============================================================

class VesuviusSliceDataset(Dataset):
    def __init__(self, volumes, labels, n_samples, patch_size=128):
        self.volumes = volumes
        self.labels = labels
        self.n_samples = n_samples
        self.patch_size = patch_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        vidx = random.randint(0, len(self.volumes) - 1)
        vol = self.volumes[vidx]
        lab = self.labels[vidx]

        Z, H, W = vol.shape

        z = random.randint(0, Z - 1)
        z_prev = max(z - 1, 0)
        z_next = min(z + 1, Z - 1)

        ps = self.patch_size
        y0 = 0 if H <= ps else random.randint(0, H - ps)
        x0 = 0 if W <= ps else random.randint(0, W - ps)

        ch0 = vol[z_prev, y0:y0+ps, x0:x0+ps]
        ch1 = vol[z,      y0:y0+ps, x0:x0+ps]
        ch2 = vol[z_next, y0:y0+ps, x0:x0+ps]

        x_np = np.stack([ch0, ch1, ch2], axis=0)

        lb_slice = lab[z, y0:y0+ps, x0:x0+ps]
        y_fg = (lb_slice == 1).astype(np.float32)
        y_ignore = (lb_slice == 2).astype(np.float32)

        x = torch.from_numpy(x_np).float()
        y = torch.from_numpy(y_fg).float().unsqueeze(0)
        ignore = torch.from_numpy(y_ignore).float().unsqueeze(0)

        return x, y, ignore

# ============================================================
# Loss (same)
# ============================================================

def loss_fn(logits, targets, ignore_mask):
    probs = torch.sigmoid(logits)
    valid_mask = 1.0 - ignore_mask

    bce = F.binary_cross_entropy(
        probs * valid_mask + 0.5 * (1 - valid_mask),
        targets,
        reduction="sum"
    ) / (valid_mask.sum() + 1e-6)

    probs_flat = (probs * valid_mask).view(logits.size(0), -1)
    targets_flat = (targets * valid_mask).view(logits.size(0), -1)

    inter = (probs_flat * targets_flat).sum(dim=1)
    denom = probs_flat.sum(dim=1) + targets_flat.sum(dim=1) + 1e-6

    dice = 1 - (2 * inter / denom)
    return bce + dice.mean(), bce.detach(), (1 - dice).detach()

# ============================================================
# Load training data (same)
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
train_ids = train_df["id"].values[:MAX_TRAIN_VOLUMES]

train_volumes = []
train_labels = []

for vid in tqdm(train_ids, desc="Loading train volumes"):
    fname = f"{vid}.tif"
    img_path = TRAIN_IMG_DIR / fname
    lbl_path = TRAIN_LABEL_DIR / fname

    if not img_path.exists() or not lbl_path.exists():
        continue

    vol = load_stack(img_path)
    lbl = load_stack(lbl_path)

    if vol.shape != lbl.shape:
        continue

    train_volumes.append(normalize_volume(vol))
    train_labels.append(lbl)

split_idx = max(1, len(train_volumes) - 3)

train_dataset = VesuviusSliceDataset(
    train_volumes[:split_idx], train_labels[:split_idx],
    n_samples=TRAIN_SAMPLES, patch_size=PATCH_SIZE
)
val_dataset = VesuviusSliceDataset(
    train_volumes[split_idx:], train_labels[split_idx:],
    n_samples=VAL_SAMPLES, patch_size=PATCH_SIZE
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
# Train (same)
# ============================================================

model = UNet2D().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_dice = 0.0
best_model_path = "/kaggle/working/best_unet2d_2p5d.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0
    train_dice_sum = 0
    train_batches = 0

    for x, y, ignore in tqdm(train_loader, desc=f"Epoch {epoch} [train]"):
        x, y, ignore = x.to(DEVICE), y.to(DEVICE), ignore.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss, bce, dice = loss_fn(logits, y, ignore)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        train_dice_sum += dice.mean().item()
        train_batches += 1

    val_loss_sum = 0
    val_dice_sum = 0
    val_batches = 0
    model.eval()

    with torch.no_grad():
        for x, y, ignore in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
            x, y, ignore = x.to(DEVICE), y.to(DEVICE), ignore.to(DEVICE)
            logits = model(x)
            loss, bce, dice = loss_fn(logits, y, ignore)

            val_loss_sum += loss.item()
            val_dice_sum += dice.mean().item()
            val_batches += 1

    train_loss = train_loss_sum / train_batches
    train_dice = train_dice_sum / train_batches
    val_loss = val_loss_sum / val_batches
    val_dice = val_dice_sum / val_batches

    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_dice={val_dice:.4f}")

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_model_path)
        print(f"New best val Dice: {best_val_dice:.4f}")

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

model.eval()

# ============================================================
# Inference (threshold upgrade applied)
# ============================================================

test_df = pd.read_csv(TEST_CSV)

with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test inference"):
        vid = row["id"]
        fname = f"{vid}.tif"
        img_path = TEST_IMG_DIR / fname

        if not img_path.exists():
            continue

        vol = normalize_volume(load_stack(img_path))
        Z, H, W = vol.shape
        pred_mask = np.zeros((Z, H, W), dtype=np.uint8)

        for z_idx in range(Z):
            z_prev = max(z_idx - 1, 0)
            z_next = min(z_idx + 1, Z - 1)

            x_np = np.stack([vol[z_prev], vol[z_idx], vol[z_next]], axis=0)
            x = torch.from_numpy(x_np).float().unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                logits = model(x)
                probs = torch.sigmoid(logits)[0, 0].cpu().numpy()

                # --------------------------------------
                # SAFE UPGRADE #2 (threshold 0.45)
                pred = (probs > THRESH).astype(np.uint8)
                # --------------------------------------

            pred_mask[z_idx] = pred

        cleaned = clean_mask(pred_mask)
        write_stack_to_zip(cleaned, zf, fname)

print("Done! submission.zip created successfully.")

ModuleNotFoundError: No module named 'numpy'